
# Notebook 15 · Predicción de devolución y margen neto post-devolución

## Pregunta de negocio
¿Qué pedidos, líneas o productos muestran mayor riesgo de devolución y cómo afecta eso al margen?

## Qué aprenderás
- Cómo etiquetar devoluciones a nivel de línea.
- Cómo combinar producto, precio y señales de fricción del cliente.
- Cómo convertir un score de devolución en decisión comercial.

## Outputs
- `nb15_return_risk_scoring.csv`
- `nb15_product_return_risk_summary.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

fact = tables["fact"].copy()
dev_on = tables["devoluciones_online"].copy()
dev_store = tables["devoluciones_tienda"].copy()
tickets = tables["tickets"].copy()
feedback = tables["feedback"].copy()

sales = fact[(fact["tipo_movimiento"] == "VENTA") & fact["line_id_std"].notna()].copy()
returned_lines = set(dev_on["order_line_id"].dropna().astype(str)).union(set(dev_store["ticket_line_id_original"].dropna().astype(str)))
sales["returned_flag"] = sales["line_id_std"].astype(str).isin(returned_lines).astype(int)

sales["price"] = sales["precio_unitario_num"]
sales["discount_pct"] = sales["descuento_pct_num"].fillna(0)
sales["margin"] = sales["margen_bruto_num"]
sales["qty"] = sales["cantidad_abs"]

sales = sales.sort_values("fecha_movimiento_dt")
sales["prior_tx_count"] = sales.groupby("customer_id").cumcount()
sales["prior_return_rate_proxy"] = sales.groupby("customer_id")["returned_flag"].transform(lambda s: s.shift().expanding().mean())

support_counts = []
neg_feedback_counts = []

for _, row in sales[["customer_id", "fecha_movimiento_dt"]].iterrows():
    cust = row["customer_id"]
    dt = row["fecha_movimiento_dt"]
    t_sub = tickets[(tickets["customer_id"] == cust) & (tickets["fecha_apertura_dt"] < dt) & (tickets["fecha_apertura_dt"] >= dt - pd.Timedelta(days=90))]
    f_sub = feedback[(feedback["customer_id"] == cust) & (feedback["fecha_feedback_dt"] < dt) & (feedback["fecha_feedback_dt"] >= dt - pd.Timedelta(days=90))]
    support_counts.append(len(t_sub))
    neg_feedback_counts.append((f_sub["sentimiento"] == "Negativo").sum())

sales["support_90d"] = support_counts
sales["neg_feedback_90d"] = neg_feedback_counts

sales[["line_id_std", "customer_id", "categoria", "price", "discount_pct", "returned_flag"]].head()


,line_id_std,customer_id,categoria,price,discount_pct,returned_flag
1829,OL-0001797,NaN,Iluminación,289.81,0.0,0
1830,OL-0001798,NaN,Muebles,693.44,0.0,0
1828,OL-0001796,NaN,Textil hogar,216.41,25.0,0
1831,OL-0001799,NaN,Iluminación,370.35,5.0,0
1973,OL-0001796,NaN,Textil hogar,216.41,25.0,0


In [3]:

features = [
    "canal_movimiento", "categoria", "subcategoria", "estado_pedido",
    "price", "discount_pct", "margin", "qty",
    "prior_tx_count", "prior_return_rate_proxy", "support_90d", "neg_feedback_90d"
]

X = sales[features]
y = sales["returned_flag"]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

model = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame({
    "metric": ["ROC_AUC", "Average_Precision", "return_rate"],
    "value": [roc_auc_score(y_test, proba_test), average_precision_score(y_test, proba_test), y.mean()]
})
metrics


,metric,value
0,ROC_AUC,0.649075
1,Average_Precision,0.287445
2,return_rate,0.083591


In [4]:

sales["return_risk_score"] = model.predict_proba(X)[:, 1]
sales["expected_margin_after_returns"] = sales["margin"] * (1 - sales["return_risk_score"])

risk_scoring = sales[[
    "line_id_std", "customer_id", "doc_id_std", "canal_movimiento", "categoria", "subcategoria",
    "price", "discount_pct", "margin", "return_risk_score", "expected_margin_after_returns",
    "support_90d", "neg_feedback_90d"
]].sort_values(["return_risk_score", "price"], ascending=[False, False])

product_summary = (
    sales.groupby(["product_id_std", "descripcion_producto", "categoria", "subcategoria"])
         .agg(
             lines=("line_id_std", "nunique"),
             avg_price=("price", "mean"),
             observed_return_rate=("returned_flag", "mean"),
             avg_predicted_return_risk=("return_risk_score", "mean"),
             avg_expected_margin_after_returns=("expected_margin_after_returns", "mean"),
         )
         .reset_index()
         .sort_values(["avg_predicted_return_risk", "observed_return_rate"], ascending=False)
)

risk_scoring.head(20), product_summary.head(15)


(     line_id_std customer_id   doc_id_std canal_movimiento     categoria    subcategoria    price  discount_pct  margin  return_risk_score  expected_margin_after_returns  support_90d  \
 1136  OL-0001109         NaN  O-202400380           ONLINE       Muebles           Sofás  1026.62           0.0  600.36              0.960                       24.01440            0   
 1137  OL-0001109         NaN  O-202400380           ONLINE       Muebles           Sofás  1026.62           0.0  600.36              0.960                       24.01440            0   
 4727   TL-000611         NaN     T-500301              POS    decoración           Camas  1214.68           0.0     NaN              0.904                            NaN            0   
 2830   TL-000611         NaN     T-500301              POS    decoración           Camas  1214.68           0.0     NaN              0.904                            NaN            0   
 831   OL-0000809       C0081  O-202400276           ONLINE   Ilu


## Lectura de negocio

Un modelo de devolución no solo sirve para “predecir”. También permite:

- revisar fichas de producto o expectativas de medidas,
- limitar ciertas ofertas donde el retorno erosiona margen,
- y redirigir la recomendación hacia productos más rentables netos.


In [5]:

path_score = mpu.export_csv(risk_scoring, "nb15_return_risk_scoring.csv")
path_prod = mpu.export_csv(product_summary, "nb15_product_return_risk_summary.csv")

print("Guardado:", path_score)
print("Guardado:", path_prod)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb15_return_risk_scoring.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb15_product_return_risk_summary.csv
